In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv(r"C:\Users\User\Desktop\water\clean_water_stats.csv")

In [8]:
# 1. Define your core clean features
FEATURE_COLS = [
    #'wtempV_°C', 
    'conductivity_of_solution_ mS_cm', 
    'oxidation_reduction_potential_ mV', 
    'turbidity_NTU', 
    'oxygen_saturation_%'
]

# List to collect processed dataframes for each sensor
processed_sensor_dfs = []

# Group by the unique station ID (976, 984, etc.)
for sensor_id, sensor_df in df.groupby('sensor_id'):
    
    # Work on a copy of this specific station's data
    s_df = sensor_df.copy()
    
    # Step A: Scale features ONLY using this station's local mean and std
    scaler = StandardScaler()
    X_sensor_scaled = scaler.fit_transform(s_df[FEATURE_COLS])
    
    # Step B: Fit PCA specifically on this station's local dynamics
    # We choose 3 components to retain local variance cleanly
    pca = PCA(n_components=3)
    X_pca = pca.fit_transform(X_sensor_scaled)
    
    # Step C: Assign local PC values back to this station's rows
    s_df['PC1'] = X_pca[:, 0]
    s_df['PC2'] = X_pca[:, 1]
    s_df['PC3'] = X_pca[:, 2]
    
    # Store local variance explained for verification
    var_explained = pca.explained_variance_ratio_
    print(f"Sensor ID {sensor_id} PCA Variance: PC1={var_explained[0]:.2%}, PC2={var_explained[1]:.2%}, PC3={var_explained[2]:.2%} | Total={sum(var_explained):.2%}")
    
    processed_sensor_dfs.append(s_df)

# Recombine all 9 sensors back into a single main DataFrame
df_pca_by_sensor = pd.concat(processed_sensor_dfs, axis=0).sort_index()

Sensor ID 976 PCA Variance: PC1=41.35%, PC2=26.29%, PC3=20.94% | Total=88.58%
Sensor ID 977 PCA Variance: PC1=43.59%, PC2=35.87%, PC3=11.28% | Total=90.74%
Sensor ID 978 PCA Variance: PC1=42.38%, PC2=26.46%, PC3=22.43% | Total=91.26%
Sensor ID 979 PCA Variance: PC1=39.74%, PC2=29.70%, PC3=17.86% | Total=87.30%
Sensor ID 980 PCA Variance: PC1=39.29%, PC2=25.70%, PC3=22.45% | Total=87.44%
Sensor ID 981 PCA Variance: PC1=49.94%, PC2=27.28%, PC3=15.83% | Total=93.06%
Sensor ID 982 PCA Variance: PC1=45.71%, PC2=22.20%, PC3=19.18% | Total=87.09%
Sensor ID 983 PCA Variance: PC1=47.76%, PC2=24.35%, PC3=19.70% | Total=91.81%
Sensor ID 984 PCA Variance: PC1=37.39%, PC2=33.79%, PC3=17.23% | Total=88.41%


In [9]:
sensor_loadings = {}
sensor_means = {}

for sensor_id, sensor_df in df.groupby('sensor_id'):
    s_df = sensor_df.copy()
    
    # 1. Scale locally & store the scaler means/stds
    scaler = StandardScaler()
    X_sensor_scaled = scaler.fit_transform(s_df[FEATURE_COLS])
    
    sensor_means[sensor_id] = pd.DataFrame({
        'mean': scaler.mean_,
        'std': scaler.scale_
    }, index=FEATURE_COLS)
    
    # 2. Fit local PCA
    pca = PCA(n_components=3)
    pca.fit(X_sensor_scaled)
    
    # 3. Extract Loadings (components_ transposed: rows=features, cols=PCs)
    loadings_df = pd.DataFrame(
        pca.components_.T, 
        columns=['PC1', 'PC2', 'PC3'], 
        index=FEATURE_COLS
    )
    
    sensor_loadings[sensor_id] = loadings_df

# --- PRINT FUNCTION FOR INSPECTION ---
def inspect_sensor_pca(sensor_id):
    print(f"\n=================== SENSOR ID: {sensor_id} ===================")
    print("\n--- 1. Original Feature Means & Stds (Local Baseline) ---")
    print(sensor_means[sensor_id].round(2))
    
    print("\n--- 2. Feature Loadings (Weights) ---")
    print(sensor_loadings[sensor_id].round(4))

In [10]:
for i in list(df['sensor_id'].unique()):
    inspect_sensor_pca(i)


=================== SENSOR ID: 976 ===================

--- 1. Original Feature Means & Stds (Local Baseline) ---
                                       mean     std
conductivity_of_solution_ mS_cm    56969.70  321.56
oxidation_reduction_potential_ mV   -117.84   31.62
turbidity_NTU                        120.72   82.13
oxygen_saturation_%                   89.26    5.80

--- 2. Feature Loadings (Weights) ---
                                      PC1     PC2     PC3
conductivity_of_solution_ mS_cm    0.0345  0.8688 -0.4937
oxidation_reduction_potential_ mV  0.4517  0.3952  0.7355
turbidity_NTU                      0.6714 -0.0544 -0.0685
oxygen_saturation_%               -0.5865  0.2932  0.4589

=================== SENSOR ID: 977 ===================

--- 1. Original Feature Means & Stds (Local Baseline) ---
                                       mean      std
conductivity_of_solution_ mS_cm    57078.28  4227.97
oxidation_reduction_potential_ mV   -151.94   112.59
turbidity_NTU         

In [6]:
loading_summary = []

for sensor_id, df_load in sensor_loadings.items():
    pc1_top = df_load['PC1'].abs().idxmax()
    pc2_top = df_load['PC2'].abs().idxmax()
    pc3_top = df_load['PC3'].abs().idxmax()
    
    loading_summary.append({
        'sensor_id': sensor_id,
        'PC1_Driver': f"{pc1_top} ({df_load.loc[pc1_top, 'PC1']:.2f})",
        'PC2_Driver': f"{pc2_top} ({df_load.loc[pc2_top, 'PC2']:.2f})",
        'PC3_Driver': f"{pc3_top} ({df_load.loc[pc3_top, 'PC3']:.2f})"
    })

pd.DataFrame(loading_summary).set_index('sensor_id')

,PC1_Driver,PC2_Driver,PC3_Driver
sensor_id,,,
976,turbidity_NTU (0.67),conductivity_of_solution_ mS_cm (0.87),oxidation_reduction_potential_ mV (0.74)
977,oxygen_saturation_% (0.67),oxidation_reduction_potential_ mV (0.75),oxidation_reduction_potential_ mV (0.66)
978,conductivity_of_solution_ mS_cm (0.68),oxidation_reduction_potential_ mV (0.87),oxygen_saturation_% (0.84)
979,oxidation_reduction_potential_ mV (0.66),turbidity_NTU (0.69),turbidity_NTU (0.67)
980,conductivity_of_solution_ mS_cm (0.67),oxidation_reduction_potential_ mV (0.79),turbidity_NTU (0.84)
981,oxygen_saturation_% (0.63),oxidation_reduction_potential_ mV (0.91),turbidity_NTU (0.77)
982,conductivity_of_solution_ mS_cm (0.58),turbidity_NTU (0.92),oxygen_saturation_% (0.88)
983,oxidation_reduction_potential_ mV (0.65),oxygen_saturation_% (0.91),conductivity_of_solution_ mS_cm (0.75)
984,oxygen_saturation_% (0.71),oxidation_reduction_potential_ mV (0.66),oxidation_reduction_potential_ mV (0.71)
